<a href="https://colab.research.google.com/github/heavenmaker024/114-2PL-Repo61271012H/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_ipynb_0402(II).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 安裝必要套件
!pip install -q google-generativeai gspread

import gspread
from datetime import datetime
import google.generativeai as genai
from google.colab import auth
from google.auth import default
from google.colab import userdata  # 引入密碼本功能
import pandas as pd
import sys

# ==========================================
# 2. 設定區：API Key 與 Google Sheet 網址
# ==========================================
# 安全地從 Colab 密碼本讀取 API 金鑰
try:
    GOOGLE_API_KEY = userdata.get('Geminiapikey')
    genai.configure(api_key=GOOGLE_API_KEY)
except userdata.SecretNotFoundError:
    print("❌ 錯誤：找不到 API 金鑰！")
    print("👉 請在 Colab 左側的「🔑 鑰匙圖示 (Secrets)」中，新增一個名為 'Geminiapikey' 的變數，並填入您的 API 金鑰。")
    sys.exit() # 終止程式

# 使用推薦的 Flash 模型
model = genai.GenerativeModel('gemini-2.5-flash')

# 設定 Google Sheet 資訊
SHEET_URL = "https://docs.google.com/spreadsheets/d/1JkzCgRWp1e0LIQ0YtIvOLm9R8ULGVyr4bVuDT6g8c2U/edit?gid=0#gid=0"
WORKSHEET_NAME = "工作表1"
REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

# ==========================================
# 3. 核心功能函式
# ==========================================
def get_user_grades():
    print("\n--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("👉 請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade_input = input(f"👉 請輸入 {subject} 的成績：")
        try:
            grade = int(grade_input)
        except ValueError:
            print("❌ 成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"✅ 已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

def get_ai_summary(grades):
    prompt_text = (
        "你是一位專業的家教。以下是學生最近的各科成績列表，"
        "請幫我根據這些成績，產出一個簡單的「學習狀況摘要」，"
        "以及針對這些科目給予「常見的學習迷思與建議」（不需要做評分，只要給予鼓勵和總結）。\n\n"
    )
    for record in grades:
        date, subject, grade = record
        prompt_text += f"- 日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n🤖 --- 正在呼叫 Gemini AI 模型生成摘要... ---")
    try:
        response = model.generate_content(prompt_text)
        return response.text
    except Exception as e:
        print(f"❌ 呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

# ==========================================
# 4. 主程式流程
# ==========================================
def main():
    try:
        print("🔄 正在連接 Google 帳號與 Google Sheets...")
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)
        print("✅ Google Sheet 連線成功！\n")

        new_grades = get_user_grades()

        if not new_grades:
            print("⚠️ 沒有輸入任何成績，程式結束。")
            return

        ws.append_rows(new_grades, value_input_option="USER_ENTERED")
        print("\n✅ --- 成績已成功寫入 Google Sheet。---")

        summary = get_ai_summary(new_grades)

        today = datetime.now().strftime('%Y-%m-%d')
        summary_row = [[today, "🤖 AI 學習摘要", summary]]
        ws.append_rows(summary_row, value_input_option="USER_ENTERED")

        print("\n✅ --- AI 摘要已成功寫入 Google Sheet。---")
        print("=" * 50)
        print("【AI 生成的摘要內容】")
        print("-" * 50)
        print(summary)
        print("=" * 50)

    except gspread.exceptions.APIError as e:
        print(f"❌ Google Sheets API 錯誤。請確認您的權限設定。")
    except Exception as e:
        print(f"❌ 發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


🔄 正在連接 Google 帳號與 Google Sheets...
✅ Google Sheet 連線成功！


--- 準備輸入成績。輸入 'q' 來停止。---


KeyboardInterrupt: Interrupted by user

In [ ]:
# 1. 安裝必要套件
!pip install -q gradio google-generativeai gspread pandas

import gradio as gr
import gspread
import pandas as pd
from datetime import datetime
import google.generativeai as genai
from google.colab import auth
from google.auth import default
from google.colab import userdata
import sys

# ==========================================
# 2. 設定區：API Key 與 Google Sheet 網址
# ==========================================
try:
    GOOGLE_API_KEY = userdata.get('Geminiapikey')
    genai.configure(api_key=GOOGLE_API_KEY)
except userdata.SecretNotFoundError:
    print("❌ 錯誤：找不到 API 金鑰！請在 Colab 左側的「🔑 Secrets」中新增 'Geminiapikey'。")
    sys.exit()

# 使用推薦的 Flash 模型
model = genai.GenerativeModel('gemini-2.5-flash')

# 設定 Google Sheet 資訊 (根據你提供的最新設定，使用工作表1)
SHEET_URL = "https://docs.google.com/spreadsheets/d/1JkzCgRWp1e0LIQ0YtIvOLm9R8ULGVyr4bVuDT6g8c2U/edit?gid=0#gid=0"
WORKSHEET_NAME = "工作表1"

# ==========================================
# 3. 核心處理函式 (與 Gradio 介面綁定)
# ==========================================
def process_grades_and_get_ai(df):
    """
    接收 Gradio 表格的成績，過濾空白資料後寫入 Google Sheet，
    再呼叫 AI 生成評語，最後將評語寫入試算表並顯示在介面上。
    """
    valid_grades = []
    today = datetime.now().strftime('%Y-%m-%d')

    # 1. 讀取並清理介面上的表格資料
    if df is not None:
        for index, row in df.iterrows():
            subject = str(row['科目']).strip()
            grade_raw = row['作業成績']

            # 過濾掉沒有填寫的空白列，確保科目存在且成績為數字
            if subject and subject.lower() not in ['none', 'nan', ''] and pd.notna(grade_raw):
                try:
                    grade = int(float(grade_raw))
                    valid_grades.append([today, subject, grade])
                except ValueError:
                    continue # 若成績輸入非數字的亂碼，則忽略該列

    if not valid_grades:
        return "⚠️ 請至少輸入一筆完整的科目與成績（成績需為數字）！", "（沒有收到有效成績，無法生成評語）"

    try:
        # 2. Google 帳號授權與連線
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)

        # 3. 將成績寫入 Google Sheet
        ws.append_rows(valid_grades, value_input_option="USER_ENTERED")

        # 4. 準備給 AI 的提示詞
        prompt_text = (
            "你是一位專業的家教。以下是學生最近的各科成績列表，"
            "請幫我根據這些成績，產出一個簡單的「學習狀況摘要」，"
            "以及針對這些科目給予「常見的學習迷思與建議」（不需要做評分，只要給予鼓勵和總結）。\n\n"
        )
        for record in valid_grades:
            prompt_text += f"- 日期：{record[0]}, 科目：{record[1]}, 成績：{record[2]}\n"

        # 5. 呼叫 Gemini AI 生成評語
        try:
            response = model.generate_content(prompt_text)
            ai_summary = response.text
        except Exception as ai_e:
            ai_summary = f"⚠️ 成績已記錄，但 AI 呼叫失敗。錯誤訊息：{ai_e}"

        # 6. 將 AI 摘要寫入 Google Sheet
        summary_row = [[today, "🤖 AI 學習摘要", ai_summary]]
        ws.append_rows(summary_row, value_input_option="USER_ENTERED")

        # 7. 回傳結果給介面
        success_msg = f"✅ 處理完成！已將 {len(valid_grades)} 筆成績與 AI 評語寫入雲端試算表。"
        return success_msg, ai_summary

    except gspread.exceptions.APIError as e:
        return f"❌ Google Sheets 連線或權限錯誤：{e}", "（因權限錯誤，未生成評語）"
    except Exception as e:
        return f"❌ 發生系統錯誤：{e}", "（發生錯誤，未生成評語）"


# ==========================================
# 4. Gradio UI 介面設計
# ==========================================
with gr.Blocks(title="AI 學習評語生成系統") as demo:
    gr.Markdown("## 📝 學生作業成績登錄與 AI 評語系統")
    gr.Markdown("💡 **操作提示**：點擊下方表格格子即可打字輸入。送出前可隨時用鍵盤修改或刪除。若列數不夠可點擊右下方的小圖示新增列。")

    with gr.Row():
        # 左側：成績輸入區
        with gr.Column(scale=1):
            grade_input = gr.Dataframe(
                headers=["科目", "作業成績"],
                datatype=["str", "number"],
                row_count=(5, "dynamic"), # 預設給 5 列，可動態增減
                col_count=(2, "fixed"),
                interactive=True,
                label="輸入成績（送出時會自動略過空白列）"
            )
            submit_btn = gr.Button("🚀 儲存成績並呼叫 AI 生成評語", variant="primary")
            status_output = gr.Textbox(label="系統執行狀態", interactive=False)

        # 右側：AI 評語顯示區
        with gr.Column(scale=1):
            ai_summary_output = gr.Textbox(
                label="🤖 AI 學習評語與建議 (完成後將顯示於此)",
                lines=15,
                interactive=False
            )

    # 綁定按鈕動作
    submit_btn.click(
        fn=process_grades_and_get_ai,
        inputs=[grade_input],
        outputs=[status_output, ai_summary_output]
    )

# 啟動介面 (share=True 會產生公開網址方便手機測試)
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fb524c1436494bef24.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fb524c1436494bef24.gradio.live


In [ ]:
# 1. 安裝必要套件
!pip install -q gradio google-generativeai gspread pandas

import gradio as gr
import gspread
import pandas as pd
from datetime import datetime
import google.generativeai as genai
from google.colab import auth
from google.auth import default
from google.colab import userdata
import sys

# ==========================================
# 2. 設定區：API Key 與 Google Sheet 網址
# ==========================================
try:
    GOOGLE_API_KEY = userdata.get('Geminiapikey')
    genai.configure(api_key=GOOGLE_API_KEY)
except userdata.SecretNotFoundError:
    print("❌ 錯誤：找不到 API 金鑰！請在 Colab 左側的「🔑 Secrets」中新增 'Geminiapikey'。")
    sys.exit()

model = genai.GenerativeModel('gemini-2.5-flash')

# 你的 Google Sheet 資訊
SHEET_URL = "https://docs.google.com/spreadsheets/d/1JkzCgRWp1e0LIQ0YtIvOLm9R8ULGVyr4bVuDT6g8c2U/edit?gid=0#gid=0"
WORKSHEET_NAME = "工作表1"

# ==========================================
# 3. 核心處理函式 (已修復 KeyError 問題)
# ==========================================
def process_grades_and_get_ai(df):
    valid_grades = []
    today = datetime.now().strftime('%Y-%m-%d')

    if df is not None:
        # 【修正核心】將 DataFrame 轉為列表 (List of lists)，避開欄位名稱遺失的 Bug
        data = df.values.tolist()

        for row in data:
            # 確保每一列至少有兩個欄位 (科目, 成績)
            if len(row) < 2:
                continue

            # 改用位置索引：row[0] 是科目，row[1] 是成績
            subject = str(row[0]).strip()
            grade_raw = row[1]

            # 過濾空白列與無效資料
            if subject and subject.lower() not in ['none', 'nan', ''] and pd.notna(grade_raw) and str(grade_raw).strip() != '':
                try:
                    grade = int(float(grade_raw))
                    valid_grades.append([today, subject, grade])
                except ValueError:
                    continue # 略過非數字的成績

    if not valid_grades:
        return "⚠️ 失敗：請至少輸入一筆完整的科目與成績（成績需為數字）！", "（沒有收到有效成績，無法生成評語）"

    try:
        # 授權與連線 Google Sheet
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)

        # 寫入成績
        ws.append_rows(valid_grades, value_input_option="USER_ENTERED")

        # 準備 AI 提示詞
        prompt_text = (
            "你是一位專業的家教。以下是學生最近的各科成績列表，"
            "請幫我根據這些成績，產出一個簡單的「學習狀況摘要」，"
            "以及針對這些科目給予「常見的學習迷思與建議」（不需要做評分，只要給予鼓勵和總結）。\n\n"
        )
        for record in valid_grades:
            prompt_text += f"- 日期：{record[0]}, 科目：{record[1]}, 成績：{record[2]}\n"

        # 呼叫 AI
        try:
            response = model.generate_content(prompt_text)
            ai_summary = response.text
        except Exception as ai_e:
            ai_summary = f"⚠️ 成績已記錄，但 AI 呼叫失敗。錯誤訊息：{ai_e}"

        # 寫入 AI 摘要
        summary_row = [[today, "🤖 AI 學習摘要", ai_summary]]
        ws.append_rows(summary_row, value_input_option="USER_ENTERED")

        success_msg = f"✅ 處理完成！已將 {len(valid_grades)} 筆成績與 AI 評語寫入雲端試算表。"
        return success_msg, ai_summary

    except gspread.exceptions.APIError as e:
        return f"❌ Google Sheets 權限錯誤：{e}", "（因權限錯誤，未生成評語）"
    except Exception as e:
        return f"❌ 發生系統錯誤：{e}", "（發生錯誤，未生成評語）"


# ==========================================
# 4. Gradio UI 介面設計
# ==========================================
with gr.Blocks(title="AI 學習評語生成系統") as demo:
    gr.Markdown("## 📝 學生作業成績登錄與 AI 評語系統")
    gr.Markdown("💡 **操作提示**：點擊下方表格格子即可打字輸入。送出前可隨時用鍵盤修改或刪除。若列數不夠可點擊右下方的小圖示新增列。")

    with gr.Row():
        with gr.Column(scale=1):
            grade_input = gr.Dataframe(
                headers=["科目", "作業成績"],
                datatype=["str", "number"],
                row_count=(5, "dynamic"),
                col_count=(2, "fixed"),
                interactive=True,
                label="輸入成績（送出時會自動略過空白列）"
            )
            submit_btn = gr.Button("🚀 儲存成績並呼叫 AI 生成評語", variant="primary")
            status_output = gr.Textbox(label="系統執行狀態", interactive=False)

        with gr.Column(scale=1):
            ai_summary_output = gr.Textbox(
                label="🤖 AI 學習評語與建議",
                lines=15,
                interactive=False
            )

    submit_btn.click(
        fn=process_grades_and_get_ai,
        inputs=[grade_input],
        outputs=[status_output, ai_summary_output]
    )

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9820f673667ca5bd05.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9820f673667ca5bd05.gradio.live
